In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.optim import AdamW
from torch.utils.data import DataLoader

from torchvision import datasets, transforms

torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
def squash(x, dim=-1, eps=1e-8):

    norm_sq = (x ** 2).sum(dim=dim, keepdim=True)
    norm = torch.sqrt(norm_sq + eps)

    scale = norm_sq / (1.0 + norm_sq)

    return scale * (x / norm)

In [ ]:
class PrimaryCaps(nn.Module):

    def __init__(
        self,
        in_channels=256,
        num_capsules=32,
        capsule_dim=8
    ):
        super().__init__()

        self.num_capsules = num_capsules
        self.capsule_dim = capsule_dim

        self.conv = nn.Conv2d(
            in_channels,
            num_capsules * capsule_dim,
            kernel_size=9,
            stride=2
        )

    def forward(self, x):

        x = self.conv(x)

        B = x.size(0)

        x = x.view(
            B,
            self.num_capsules,
            self.capsule_dim,
            x.size(2),
            x.size(3)
        )

        x = x.permute(0, 1, 3, 4, 2).contiguous()

        x = x.view(B, -1, self.capsule_dim)

        return squash(x)

In [ ]:
class DigitCaps(nn.Module):

    def __init__(
        self,
        num_capsules=10,
        num_routes=1152,
        in_dim=8,
        out_dim=16,
        routing_iters=3,
        temperature=1.0
    ):
        super().__init__()

        self.num_capsules = num_capsules
        self.num_routes = num_routes
        self.routing_iters = routing_iters
        self.temperature = temperature

        self.W = nn.Parameter(
            0.01 * torch.randn(
                1,
                num_routes,
                num_capsules,
                out_dim,
                in_dim
            )
        )

    def forward(self, x):

        B = x.size(0)

        x = x.unsqueeze(2).unsqueeze(-1)

        W = self.W.expand(B, -1, -1, -1, -1)

        u_hat = torch.matmul(W, x).squeeze(-1)

        b_ij = torch.zeros(
            B,
            self.num_routes,
            self.num_capsules,
            1,
            device=x.device
        )

        for i in range(self.routing_iters):

            c_ij = F.softmax(
                b_ij / self.temperature,
                dim=2
            )

            s_j = (c_ij * u_hat).sum(
                dim=1,
                keepdim=True
            )

            v_j = squash(s_j, dim=-1)

            if i < self.routing_iters - 1:

                agreement = (
                    u_hat * v_j
                ).sum(dim=-1, keepdim=True)

                agreement = torch.clamp(
                    agreement,
                    -10,
                    10
                )

                b_ij = b_ij + agreement

        return v_j.squeeze(1)

In [ ]:
class CapsNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(
                1,
                256,
                kernel_size=9,
                stride=1
            ),
            nn.ReLU(inplace=True)
        )

        self.primary_caps = PrimaryCaps()

        self.digit_caps = DigitCaps(
            routing_iters=3
        )

        self.decoder = nn.Sequential(

    nn.Linear(16 * 10, 1024),
    nn.ReLU(inplace=True),

    nn.Linear(1024, 2048),
    nn.ReLU(inplace=True),

    nn.Linear(2048, 4096),
    nn.ReLU(inplace=True),

    nn.Linear(4096, 784),
    nn.Sigmoid()
)

    def forward(self, x, labels=None):

        B = x.size(0)

        x = self.conv1(x)

        x = self.primary_caps(x)

        digit_caps = self.digit_caps(x)

        lengths = torch.norm(
            digit_caps,
            dim=-1
        )

        if labels is None:

            preds = lengths.argmax(dim=1)

            labels = F.one_hot(
                preds,
                num_classes=10
            ).float()

        masked = digit_caps * labels.unsqueeze(-1)

        reconstruction = self.decoder(
            masked.view(B, -1)
        )

        return digit_caps, lengths, reconstruction

In [ ]:
class CapsuleLoss(nn.Module):

    def __init__(
        self,
        m_plus=0.9,
        m_minus=0.1,
        lambda_=0.5,
        recon_scale=0.01
    ):
        super().__init__()

        self.m_plus = m_plus
        self.m_minus = m_minus
        self.lambda_ = lambda_
        self.recon_scale = recon_scale

    def forward(
        self,
        images,
        labels,
        lengths,
        reconstruction
    ):

        left = F.relu(
            self.m_plus - lengths
        ) ** 2

        right = F.relu(
            lengths - self.m_minus
        ) ** 2

        margin_loss = (
            labels * left +
            self.lambda_ * (1.0 - labels) * right
        )

        margin_loss = margin_loss.sum(dim=1).mean()

        reconstruction_loss = F.mse_loss(
            reconstruction,
            images.view(images.size(0), -1)
        )

        total_loss = (
            margin_loss +
            self.recon_scale * reconstruction_loss
        )

        return total_loss

In [ ]:
train_transform = transforms.ToTensor()

test_transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
model = CapsNet().to(DEVICE)

criterion = CapsuleLoss()

optimizer = AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)



print("Model Ready")

In [ ]:
def train_one_epoch(epoch):

    model.train()

    running_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(DEVICE)

        labels_onehot = F.one_hot(
            labels,
            num_classes=10
        ).float().to(DEVICE)

        optimizer.zero_grad()

        # forward pass
        _, lengths, reconstruction = model(
            images,
            labels_onehot
        )

        # loss
        loss = criterion(
            images,
            labels_onehot,
            lengths,
            reconstruction
        )

        # backward pass
        loss.backward()

        # gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        # update weights
        optimizer.step()

        running_loss += loss.item()

        if batch_idx % 100 == 0:

            print(
                f"Epoch {epoch} | "
                f"Batch {batch_idx} | "
                f"Loss {loss.item():.4f}"
            )

    return running_loss / len(train_loader)

In [ ]:
def evaluate():

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            _, lengths, _ = model(images)

            preds = lengths.argmax(dim=1)

            correct += (preds == labels).sum().item()

            total += labels.size(0)

    acc = 100.0 * correct / total

    print(f"\nAccuracy: {acc:.2f}%")

    return acc

In [ ]:
EPOCHS = 5

best_acc = 0.0

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(epoch)

    acc = evaluate()

    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc

    print(
        f"\nEpoch {epoch} Complete | "
        f"Train Loss: {train_loss:.4f} | "
        f"Best Accuracy: {best_acc:.2f}%\n"
    )

In [ ]:
import matplotlib.pyplot as plt

model.eval()

images, labels = next(iter(test_loader))

images_device = images.to(DEVICE)

with torch.no_grad():

    _, _, reconstruction = model(images_device)

reconstruction = reconstruction.cpu().view(-1, 28, 28)

num_images = 5

plt.figure(figsize=(20, 4))

for i in range(num_images):

    # original
    plt.subplot(2, num_images, i + 1)

    plt.imshow(images[i].squeeze(0), cmap="gray")

    plt.title(f"Original: {labels[i].item()}")

    plt.axis("off")

    # reconstructed
    plt.subplot(2, num_images, num_images + i + 1)

    plt.imshow(reconstruction[i], cmap="gray")

    plt.title("Reconstructed")

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
device = 'cuda'
# 1. Get data
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

# 2. Predict and Reconstruct
model.eval()
with torch.no_grad():
    v, lengths, reconstructions = model(images)
    preds = lengths.argmax(dim=1)

# 3. Process for display
images = images.cpu().numpy()
reconstructions = reconstructions.cpu().view(-1, 28, 28).numpy()
preds = preds.cpu().numpy()
labels = labels.cpu().numpy()

# 4. Plot top 5 results
fig, axes = plt.subplots(5, 2, figsize=(8, 20))
for i in range(5):
    # Original
    axes[i, 0].imshow(images[i].squeeze(), cmap='gray')
    axes[i, 0].set_title(f"Target: {labels[i]} | Pred: {preds[i]}")
    axes[i, 0].axis('off')

    # Reconstruction
    axes[i, 1].imshow(reconstructions[i], cmap='gray')
    axes[i, 1].set_title("Reconstructed")
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import os
os.makedirs("model", exist_ok=True)

In [ ]:
torch.save(
    model.state_dict(),
    f"model/capsnet_weights.pth"
)

print("Weights saved")

In [ ]:
torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "best_acc": best_acc
}, f"model/capsnet_checkpoint.pth")

print("Checkpoint saved")